# 01 Characterisation

This notebook characterises how RPF/sign-error labels appear in final Alpha and Beta. Gamma is intentionally left out of final characterisation artefacts because it is handled as the forecast-impact case study.


## 1. Imports And Paths

Load the v2 config and confirm that the notebook will read from `dataset/final/`.


In [1]:
from pathlib import Path
import sys

# Keep notebook imports stable whether the notebook is run from JupyterLab,
# VS Code, or the repository root.
article_root = Path.cwd()
while article_root.name != "2_journal_article":
    if article_root.parent == article_root:
        raise RuntimeError("Could not locate publication/2_journal_article")
    article_root = article_root.parent
notebook_dir = article_root / "notebooks"
if str(notebook_dir) not in sys.path:
    sys.path.insert(0, str(notebook_dir))

import _experiment_helpers as h

cfg = h.load_config(article_root)
paths = h.article_paths(article_root, cfg)
h.ensure_output_dirs(paths)
print(f"Article root: {article_root}")
print(f"Config schema: {cfg['schema_version']}")
print(f"Output root: {paths.outputs}")

print(article_root / cfg["paths"]["alpha_dataset_path"])
print(article_root / cfg["paths"]["beta_dataset_path"])


Article root: c:\Users\samha\Documents\PyNRPF\publication\2_journal_article
Config schema: journal_v2
Output root: c:\Users\samha\Documents\PyNRPF\publication\2_journal_article\outputs
c:\Users\samha\Documents\PyNRPF\publication\2_journal_article\dataset\final\dataset_alpha.parquet
c:\Users\samha\Documents\PyNRPF\publication\2_journal_article\dataset\final\dataset_beta.parquet


## 2. Load Final Alpha And Beta

The helper validates the seven-column schema, parses wall-clock timestamps, and derives analysis-only columns such as reference net load.


In [2]:
alpha = h.load_dataset(article_root, cfg, "alpha")
beta = h.load_dataset(article_root, cfg, "beta")
[h.dataset_summary(alpha, "Alpha"), h.dataset_summary(beta, "Beta")]


[{'dataset': 'Alpha',
  'n_rows': 1011264,
  'n_sites': 10,
  'min_timestamp': '2021-11-01 00:15:00',
  'max_timestamp': '2024-09-30 00:00:00',
  'n_dates': 1065,
  'null_net_load_MW': 3000,
  'null_solar_MW': 1040,
  'positive_label_interval': 47980,
  'positive_label_day_site_days': 3423},
 {'dataset': 'Beta',
  'n_rows': 280800,
  'n_sites': 8,
  'min_timestamp': '2023-10-01 00:00:00',
  'max_timestamp': '2024-09-30 23:45:00',
  'n_dates': 366,
  'null_net_load_MW': 4394,
  'null_solar_MW': 800,
  'positive_label_interval': 10604,
  'positive_label_day_site_days': 557}]

## 3. Build Characterisation Outputs

This writes detailed intermediates and a compact set of final Alpha/Beta tables and figures.


In [3]:
outputs = h.run_characterisation(article_root)
outputs["occurrence_dataset"]


,dataset,n_sites,total_site_days,rpf_site_days,rpf_site_day_pct,rpf_intervals
0,Alpha,10,10643,3423,32.161984,47980
1,Beta,8,2928,557,19.023224,10604


## 4. Inspect Important Tables

Use these previews to sanity-check the story before deciding whether to run further diagnostics or regenerate figures.


In [4]:
display(outputs["occurrence"].sort_values(["dataset", "rpf_days"], ascending=[True, False]).head(20))
display(outputs["event_summary"])
display(outputs["events"].sort_values("duration_minutes", ascending=False).head(10))


,dataset,substation_id,total_days,rpf_days,rpf_day_pct,rpf_intervals
5,Alpha,syn_F,1063,856,80.526811,14105
4,Alpha,syn_E,1063,712,66.980245,9965
6,Alpha,syn_G,1063,656,61.712135,9557
2,Alpha,syn_C,1065,632,59.342723,7984
9,Alpha,syn_J,1065,389,36.525822,4329
8,Alpha,syn_I,1064,151,14.191729,1898
1,Alpha,syn_B,1065,27,2.535211,142
0,Alpha,syn_A,1065,0,0.000000,0
3,Alpha,syn_D,1065,0,0.000000,0
7,Alpha,syn_H,1065,0,0.000000,0


,dataset,n_events,duration_minutes_mean,duration_minutes_median,duration_minutes_max,min_reference_net_load_MW_min,max_raw_net_load_MW_max
0,Alpha,5876,122.481280,60.0,540,-7.075031,7.075031
1,Beta,574,277.108014,285.0,480,-8.245311,8.245311


,dataset,substation_id,date,start_timestamp,end_timestamp,duration_minutes,n_intervals,min_reference_net_load_MW,max_raw_net_load_MW
3453,Alpha,syn_F,2023-12-22,2023-12-22 08:30:00,2023-12-22 17:15:00,540,36,-5.500293,5.500293
2837,Alpha,syn_F,2022-11-18,2022-11-18 08:45:00,2022-11-18 17:00:00,510,34,-4.545086,4.545086
2871,Alpha,syn_F,2022-12-08,2022-12-08 08:45:00,2022-12-08 17:00:00,510,34,-4.072858,4.072858
2841,Alpha,syn_F,2022-11-22,2022-11-22 08:45:00,2022-11-22 17:00:00,510,34,-4.886354,4.886354
2884,Alpha,syn_F,2022-12-15,2022-12-15 08:45:00,2022-12-15 17:00:00,510,34,-4.607662,4.607662
4617,Alpha,syn_G,2023-11-10,2023-11-10 08:45:00,2023-11-10 16:45:00,495,33,-3.552246,3.552246
2840,Alpha,syn_F,2022-11-21,2022-11-21 08:45:00,2022-11-21 16:45:00,495,33,-4.056299,4.056299
2880,Alpha,syn_F,2022-12-13,2022-12-13 08:45:00,2022-12-13 16:45:00,495,33,-4.507669,4.507669
2843,Alpha,syn_F,2022-11-24,2022-11-24 08:45:00,2022-11-24 16:45:00,495,33,-4.106404,4.106404
3431,Alpha,syn_F,2023-12-04,2023-12-04 09:00:00,2023-12-04 17:00:00,495,33,-3.657615,3.657615
